# PD-001 — short performance pilot

Run All in the existing Kaggle Notebook with Internet and one T4. No secrets or manual source fields are needed. Default: M0 and P50-F16, B0 versus LIGHT only, one warmup + five measured steps, one repeat: **four rounds**. No automatic runtime deadline, process kill or timed pause is installed.

Resource use is controlled by workload: 24 warmup/measured optimizer steps, four isolated profiler steps, two LIGHT diagnostic steps and a two-step synthetic checkpoint rehearsal (32 optimizer steps in the default first session, plus setup and evaluation). The previous default executed 42 rounds and 1050 warmup/measured steps alone. Intermediate ablations and repeat sweeps are not run by default.

Plan for a session well below two hours: the supplied T4 log suggests approximately 5–15 minutes for this reduced workload, with variation from setup, profiling and GPU load. This is an estimate, not a measured timing guarantee. Each completed round is saved immediately; workers run to completion. Short measurements are engineering pilot evidence, not a statistical confirmation. No full training or confirmation data are authorized. Save Version to retain Output.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json
SOURCE_COMMIT = '6a7e818d2b29fc482e93eac3cb85cfe12f891ca3'
BASELINE_COMMIT = '2e77369b0df1ce704863a815dacc48301736dde0'
WARMUP, STEPS, REPEATS = 1, 5, 1
RESTORE_FROM = ""  # optional: portable directory from a RETAINED previous Kaggle Output
CASES = ['B0','LIGHT']
WORK = Path('/kaggle/working/pd001-performance-' + SOURCE_COMMIT[:12])
REL = 'research/projects/cellular-language-models/experiments/006-path-first-clm/extension-parameter-depth'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # both arms measured sequentially on the same GPU
if SOURCE_COMMIT.startswith('__'):
    raise RuntimeError('Import the committed performance notebook with its pinned source')
WORK.mkdir(parents=True,exist_ok=True)
def checkout(name, commit):
    root = WORK/name
    if (root/'.git').exists():
        head = subprocess.check_output(['git','rev-parse','HEAD'],cwd=root,text=True).strip()
        if head != commit:
            raise RuntimeError('Existing code checkout identity mismatch; preserve outputs')
        return root
    root.mkdir(parents=True,exist_ok=False)
    def git(*args):
        print('[Setup] ' + ' '.join(args),flush=True)
        subprocess.run(['git',*args],cwd=root,check=True)
    git('init','-b','performance')
    git('remote','add','origin','https://github.com/ArcheLabs/mini-cells.git')
    git('sparse-checkout','init','--no-cone')
    git('sparse-checkout','set','--no-cone','/pyproject.toml','/src/','/shared/python/',
        '/research/projects/cellular-language-models/experiments/006-path-first-clm/',
        '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/')
    git('fetch','--depth=1','--filter=blob:none','origin',commit)
    git('checkout','--detach','FETCH_HEAD')
    return root
ROOT = checkout('repaired-code', SOURCE_COMMIT)
BASELINE = checkout('baseline-code', BASELINE_COMMIT)

ENTRY = ROOT/REL/'run_pd001.py'


In [ ]:
if not RESTORE_FROM:
    ENTRY = ROOT/REL/'run_pd001.py'
    OUTPUT = WORK/'measurements'
    if OUTPUT.exists():
        raise RuntimeError('Performance evidence already exists; download it or start a fresh output folder')
    command = [sys.executable,str(ENTRY),'--mode','perf-audit','--all-arms',
               '--baseline-root',str(BASELINE),'--device','cuda:0',
               '--allow-gpu-microbenchmark','--warmup',str(WARMUP),'--steps',str(STEPS),
               '--repeats',str(REPEATS),'--cases',*CASES,'--output',str(OUTPUT)]
    print('Sequential synthetic microbenchmarks; no full training',flush=True)
    subprocess.run(command,cwd=ROOT,check=True)
    report = json.loads((OUTPUT/'ablation.json').read_text())
    print(json.dumps(report['summary'],indent=2))
    print('Download/Save Version output:',OUTPUT)
    print('Optional engineering audit complete. User-authorized development training uses notebook.ipynb; no audit or persistence prerequisite.')


## Cross-session checkpoint rehearsal

After the benchmark, this notebook creates a bounded **two-step synthetic** P50-F16 checkpoint. Save Version to retain it. In a new cloud session attach that Notebook Output as input and set `RESTORE_FROM` to the `portable` directory shown below. Run All will skip the benchmark, restore the checkpoint, perform the remaining two steps and compare against a fresh uninterrupted four-step control. This is still engineering validation, not a language experiment. A local process restart is not evidence that Kaggle retained the files; record both Kaggle version links when reviewing the gate.


In [ ]:
PHASE = 'resume' if RESTORE_FROM else 'pause'
REHEARSAL_OUTPUT = WORK/('resume-rehearsal-' + PHASE)
cmd = [sys.executable,str(ENTRY),'--mode','resume-audit','--arm','P50-F16',
       '--source-commit',SOURCE_COMMIT,'--device','cuda:0','--allow-gpu-microbenchmark',
       '--rehearsal-phase',PHASE,'--output',str(REHEARSAL_OUTPUT)]
if RESTORE_FROM:
    cmd += ['--restore-from',RESTORE_FROM]
subprocess.run(cmd,cwd=ROOT,check=True)
print((REHEARSAL_OUTPUT/'resume-audit.json').read_text())
if PHASE == 'pause':
    print('Save Version must retain this Output. In a NEW cloud session, attach that Output, set RESTORE_FROM to its portable directory, and Run All again. This verifies persistence rather than just reusing /working.')
